## pipeline_module.py

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from transformers import pipeline as hf_pipeline

# ----------------------------------------------------------------------
# 1. NLP 모듈: 텍스트 감성 점수 추출
# ----------------------------------------------------------------------
class NLPExtractor:
    """Hugging Face 기반의 KR-FINBERT를 사용하여 감성 점수를 추출하는 모듈."""
    def __init__(self, model_name="snunlp/KR-FINBERT-SC"):
        print(f"✅ NLP 모듈 초기화: {model_name} 로드 중...")
        # 'text-classification' 파이프라인 로드
        self.classifier = hf_pipeline("text-classification", model=model_name)
        print("✅ NLP 모듈 로드 완료.")

    def extract_sentiment(self, texts: list[str]) -> pd.DataFrame:
        """
        텍스트 리스트를 입력받아 감성 점수를 반환합니다.

        반환: pd.DataFrame (label, score)
        """
        if not texts:
            return pd.DataFrame(columns=['label', 'score'])
        
        # FinBERT 추론 실행
        results = self.classifier(texts, truncation=True)
        
        scores = [{
            'sentiment_label': res['label'], 
            'sentiment_score': res['score'] * (1 if res['label'] == '긍정' else -1) # 긍정: +, 부정: -
        } for res in results]
        
        return pd.DataFrame(scores)

# ----------------------------------------------------------------------
# 2. GCN 모듈: Node Embedding 추출
# ----------------------------------------------------------------------
class GCNFeatureExtractor(torch.nn.Module):
    """
    GCN 모델 구조 정의. 이 모델의 최종 출력 (32차원 벡터)이
    'Node Embedding'으로 XGBoost/LightGBM으로 전달됩니다.
    """
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        print(f"✅ GCN 초기화: 입력={in_channels}, 임베딩 차원={out_channels}")

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GCNPipeline:
    """GCN 모델을 통해 초기 피처와 관계로부터 Node Embedding을 추출하는 파이프라인."""
    def __init__(self, in_dim: int, out_dim: int = 32, model_path: str = None):
        self.model = GCNFeatureExtractor(in_channels=in_dim, hidden_channels=64, out_channels=out_dim)
        
        # 학습된 가중치 로드 (실제 운영 환경)
        if model_path:
            try:
                self.model.load_state_dict(torch.load(model_path))
                self.model.eval()
            except FileNotFoundError:
                print(f"경고: 학습된 GCN 가중치 파일({model_path})을 찾을 수 없습니다. 무작위 가중치로 실행됩니다.")

    def _create_graph_data(self, features_df: pd.DataFrame, relations_df: pd.DataFrame) -> Data:
        """DataFrame을 PyG Data 객체로 변환합니다."""
        
        # features_df의 인덱스를 노드 ID로 사용한다고 가정합니다.
        # 1. Node Features (초기 피처) 생성
        features_tensor = torch.tensor(features_df.values, dtype=torch.float)
        
        # 2. Edge Index 생성 (종목 관계)
        # DataFrame 인덱스(0부터 시작)를 사용해야 합니다.
        node_map = {ticker: i for i, ticker in enumerate(features_df.index)}
        
        source_nodes = relations_df['from_ticker'].map(node_map).values
        target_nodes = relations_df['to_ticker'].map(node_map).values
        
        edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
        
        data = Data(x=features_tensor, edge_index=edge_index)
        print(f"✅ 그래프 데이터 생성 완료. 노드 수: {data.num_nodes}, 엣지 수: {data.num_edges}")
        return data

    def extract_embedding(self, initial_features: pd.DataFrame, relations_df: pd.DataFrame) -> pd.DataFrame:
        """
        GCN 추론을 통해 Node Embedding을 추출하고 DataFrame 형태로 반환합니다.
        """
        graph_data = self._create_graph_data(initial_features, relations_df)
        
        self.model.eval() # 추론 모드
        with torch.no_grad():
            # [노드 수, out_dim] 형태의 Node Embedding 텐서
            node_embeddings = self.model(graph_data.x, graph_data.edge_index).numpy()

        # 결과를 DataFrame으로 변환하여 반환
        embedding_df = pd.DataFrame(
            node_embeddings,
            index=initial_features.index,
            columns=[f'gcn_emb_{i}' for i in range(self.model.conv2.out_channels)]
        )
        print(f"✅ GCN Node Embedding 추출 완료. 형태: {embedding_df.shape}")
        return embedding_df

# ----------------------------------------------------------------------
# 3. XGBoost 모듈: 최종 학습 및 예측
# ----------------------------------------------------------------------
class XGBoostModel:
    """최종 피처를 입력받아 XGBoost를 학습/추론하는 모듈."""
    def __init__(self, params: dict):
        self.model = xgb.XGBClassifier(**params, random_state=42)

    def train_and_evaluate(self, X: pd.DataFrame, y: pd.Series, test_size=0.2):
        """데이터를 분할하고 모델을 학습 및 평가합니다."""
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=test_size,    
            random_state=42,  
            stratify=y 
        )
        
        print(f"✅ XGBoost 학습 데이터 크기: {len(X_train)}")
        self.model.fit(X_train, y_train) 
        
        y_pred = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        print(f"--- XGBoost 결과 ---")
        print(f"테스트 데이터 정확도 (Accuracy): {accuracy:.4f}")
        print(f"----------------------")
        return accuracy

## main.py (주윤)

In [5]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def run_pipeline():
    """
    파이프라인의 각 모듈을 순차적으로 실행하고, 데이터를 연결하여 최종 XGBoost에 전달합니다.
    """
    print("=============================================")
    print("         🚀 금융 AI 파이프라인 시작         ")
    print("=============================================")
    
    # ----------------------------------------------------------------------
    # A. 더미 데이터 생성 (실제 DB 로드 대체)
    # ----------------------------------------------------------------------
    TICKER_COUNT = 100
    TIME_STEPS = 100
    FEATURE_COUNT = 10 
    
    # NLP 입력 데이터 (뉴스 기사 리스트)
    nlp_texts = [
        "엔비디아가 역대 최대 분기 실적을 기록하며 주가가 급등했다. AI 수요 폭발.",
        "경쟁사의 리콜 이슈로 인해 주가가 급락하며 시장에 부정적인 영향을 주었다.",
        "시장에 특별한 변화는 없었으나, 유가 급등으로 인해 운송주가 상승했다."
    ]
    
    # GCN 입력 데이터 (초기 피처 & 관계)
    ticker_list = [f'TKR{i:03d}' for i in range(TICKER_COUNT)]
    
    initial_features = pd.DataFrame(
        np.random.rand(TICKER_COUNT, FEATURE_COUNT),
        index=ticker_list,
        columns=[f'initial_feature_{i}' for i in range(FEATURE_COUNT)]
    )

    relations = pd.DataFrame({
        'from_ticker': np.random.choice(ticker_list, 1000),
        'to_ticker': np.random.choice(ticker_list, 1000),
        'weight': np.random.rand(1000) 
    })
    
    # XGBoost 학습 데이터 (Feature Store 시뮬레이션 - N_SAMPLES = 100 * 100 = 10000)
    N_SAMPLES = TICKER_COUNT * TIME_STEPS
    FINAL_FEATURE_COUNT = 120 
    X_final = pd.DataFrame(
        np.random.randn(N_SAMPLES, FINAL_FEATURE_COUNT),
        columns=[f'feature_{i}' for i in range(FINAL_FEATURE_COUNT)]
    )
    y_final = pd.Series(np.random.randint(0, 2, N_SAMPLES), name='target')

    # ----------------------------------------------------------------------
    # B. 파이프라인 실행
    # ----------------------------------------------------------------------
    
    # 1. NLP 모듈 실행
    nlp_module = NLPExtractor()
    sentiment_results = nlp_module.extract_sentiment(nlp_texts)
    print("\n[단계 1. NLP 결과 (샘플)]")
    print(sentiment_results)
    
    # 2. GCN 모듈 실행
    # GCN 입력 차원은 initial_features의 컬럼 수
    gcn_module = GCNPipeline(in_dim=FEATURE_COUNT, out_dim=32)
    node_embeddings_df = gcn_module.extract_embedding(
        initial_features=initial_features, 
        relations_df=relations
    )
    print("\n[단계 2. GCN Embedding 결과 (샘플)]")
    print(node_embeddings_df.head(3))
    
    # GCN Embedding 결과는 Feature Store에 저장될 최종 피처에 통합되어야 함.
    # (여기서는 X_final에 이미 통합되었다고 가정하고 XGBoost로 넘어갑니다.)
    
    # 3. XGBoost 모듈 실행 (더미 데이터로 학습 및 평가)
    xgb_params = {
        'objective': 'binary:logistic', 'eval_metric': 'logloss',
        'eta': 0.1, 'max_depth': 6, 'use_label_encoder': False
    }
    xgb_module = XGBoostModel(params=xgb_params)
    
    print("\n[단계 3. XGBoost 실행]")
    xgb_module.train_and_evaluate(X=X_final, y=y_final)
    
    print("=============================================")
    print("         ✅ 파이프라인 전체 실행 완료         ")
    print("=============================================")

if __name__ == '__main__':
    run_pipeline()

         🚀 금융 AI 파이프라인 시작         
✅ NLP 모듈 초기화: snunlp/KR-FINBERT-SC 로드 중...


config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

c:\Users\user\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--snunlp--KR-FINBERT-SC. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For b

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


✅ NLP 모듈 로드 완료.

[단계 1. NLP 결과 (샘플)]
  sentiment_label  sentiment_score
0        positive        -0.999885
1        negative        -0.999695
2         neutral        -0.986806
✅ GCN 초기화: 입력=10, 임베딩 차원=32
✅ 그래프 데이터 생성 완료. 노드 수: 100, 엣지 수: 1000
✅ GCN Node Embedding 추출 완료. 형태: (100, 32)

[단계 2. GCN Embedding 결과 (샘플)]
        gcn_emb_0  gcn_emb_1  gcn_emb_2  gcn_emb_3  gcn_emb_4  gcn_emb_5  \
TKR000   0.384918   0.120890   0.188861  -0.339133  -0.159044   0.102429   
TKR001   0.485882   0.159524   0.267995  -0.419132  -0.208954   0.129129   
TKR002   0.486955   0.137881   0.292608  -0.444248  -0.225764   0.156542   

        gcn_emb_6  gcn_emb_7  gcn_emb_8  gcn_emb_9  ...  gcn_emb_22  \
TKR000   0.107258  -0.004147  -0.191318   0.377969  ...   -0.281702   
TKR001   0.133781  -0.006880  -0.240353   0.520326  ...   -0.368241   
TKR002   0.125632  -0.020776  -0.235608   0.541921  ...   -0.408352   

        gcn_emb_23  gcn_emb_24  gcn_emb_25  gcn_emb_26  gcn_emb_27  \
TKR000    0.226520    0

C:\Users\user\AppData\Local\Temp\ipykernel_21488\2358913780.py:90: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
c:\Users\user\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:50:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


--- XGBoost 결과 ---
테스트 데이터 정확도 (Accuracy): 0.5100
----------------------
         ✅ 파이프라인 전체 실행 완료         
